# Camoufox × 小红书 发布流程

按顺序逐格运行即可。浏览器对象跨格复用，不要重启 Kernel。

| 格 | 功能 |
|---|---|
| 1 | 导入 & 常量 |
| 2 | 启动服务器（后台线程） |
| 3 | 连接浏览器 & 恢复登录态 |
| 4 | 打开小红书发布页 |
| 5 | 手动登录 → 保存登录态（已登录可跳过） |
| 6 | 填写发布内容 |
| 7 | 关闭页面 |
| 8 | 关闭服务器 |

In [ ]:
# ── Cell 1 · 导入 & 常量 ─────────────────────────────────────────────
import base64
import json
import re
import subprocess
import sys
import time
from pathlib import Path
from threading import Thread

import orjson
from camoufox.server import LAUNCH_SCRIPT, get_nodejs, to_camel_case_dict
from camoufox.utils import launch_options
from playwright.sync_api import sync_playwright

URL_FILE   = Path("server_url.txt")
STATE_FILE = Path("xhs_state.json")
PUBLISH_URL = "https://creator.xiaohongshu.com/publish/publish"
NOTE_TITLE  = "测试标题 - 自动化测试"
NOTE_BODY   = "这是一篇自动化测试笔记，仅用于流程验证，不会实际发布。"

# 跨格共享的对象（由后续格赋值）
_server_proc = None
_playwright  = None
_browser     = None
_context     = None
_page        = None

print("✓ 导入完成")

In [ ]:
# ── Cell 2 · 启动服务器（后台线程）───────────────────────────────────
WS_PATTERN = re.compile(r"ws://\S+")

def _remove_none(d):
    return {k: v for k, v in d.items() if v is not None}

def _pipe_and_capture(stream, prefix, result):
    for raw in stream:
        line = raw if isinstance(raw, str) else raw.decode("utf-8", errors="replace")
        print(prefix + line, end="")
        if not result:
            m = WS_PATTERN.search(line)
            if m:
                result.append(m.group())

config  = _remove_none(launch_options(headless=False))
payload = base64.b64encode(orjson.dumps(to_camel_case_dict(config))).decode()
nodejs  = get_nodejs()

_server_proc = subprocess.Popen(
    [nodejs, str(LAUNCH_SCRIPT)],
    cwd=Path(nodejs).parent / "package",
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
)
_server_proc.stdin.write(payload)
_server_proc.stdin.close()

_ws_result = []
Thread(target=_pipe_and_capture, args=(_server_proc.stdout, "",       _ws_result), daemon=True).start()
Thread(target=_pipe_and_capture, args=(_server_proc.stderr, "[ERR] ", []),         daemon=True).start()

# 等待 WebSocket 地址出现
deadline = time.time() + 30
while not _ws_result and time.time() < deadline:
    if _server_proc.poll() is not None:
        raise RuntimeError("服务器进程意外退出")
    time.sleep(0.2)

if not _ws_result:
    _server_proc.terminate()
    raise TimeoutError("超时：未能获取 WebSocket 地址")

ws_url = _ws_result[0]
URL_FILE.write_text(ws_url, encoding="utf-8")
print(f"\n✓ 服务器已启动: {ws_url}")

In [ ]:
# ── Cell 3 · 连接浏览器 & 恢复登录态 ────────────────────────────────
ws_url = URL_FILE.read_text(encoding="utf-8").strip()

_playwright = sync_playwright().start()
_browser    = _playwright.firefox.connect(ws_url)
_context    = _browser.new_context()

if STATE_FILE.exists():
    state   = json.loads(STATE_FILE.read_text(encoding="utf-8"))
    cookies = state.get("cookies", [])
    if cookies:
        _context.add_cookies(cookies)
    origins = state.get("origins", [])
    if origins:
        _tmp = _context.new_page()
        for origin in origins:
            entries = origin.get("localStorage", [])
            if not entries:
                continue
            try:
                _tmp.goto(origin["origin"], wait_until="domcontentloaded", timeout=8000)
                for item in entries:
                    _tmp.evaluate("([k,v]) => localStorage.setItem(k,v)", [item["name"], item["value"]])
            except Exception:
                pass
        _tmp.close()
    print(f"✓ 已从 {STATE_FILE.name} 恢复登录态")
else:
    print("ℹ 未找到已保存的登录态，需要手动登录（见 Cell 5）")

In [ ]:
# ── Cell 4 · 打开小红书发布页 ─────────────────────────────────────────
_page = _context.new_page()
_page.goto(PUBLISH_URL, wait_until="domcontentloaded")
_page.wait_for_timeout(2000)

_is_login = any(k in _page.url for k in ("login", "signin", "sign_in"))
print(f"当前 URL  : {_page.url}")
print(f"页面标题  : {_page.title()}")
if _is_login:
    print("\n⚠ 检测到未登录 → 请在浏览器中手动登录，完成后运行 Cell 5")
else:
    print("✓ 已登录，可直接运行 Cell 6 填写内容（跳过 Cell 5）")

In [ ]:
# ── Cell 5 · 手动登录完成 → 保存登录态（已登录可跳过）───────────────
# 在浏览器中完成登录后运行此格
_page.goto(PUBLISH_URL, wait_until="domcontentloaded")
_page.wait_for_timeout(2000)

_is_login = any(k in _page.url for k in ("login", "signin", "sign_in"))
if _is_login:
    print("⚠ 仍在登录页，请先完成登录再运行本格")
else:
    state = _context.storage_state()
    STATE_FILE.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"✓ 登录态已保存至 {STATE_FILE}，下次无需再次登录")

In [ ]:
# ── Cell 6 · 填写发布内容 ─────────────────────────────────────────────
# 切换到「文字」Tab
text_tab = _page.locator('div[class*="tab"]:has-text("文字"), button:has-text("文字")')
if text_tab.count() > 0:
    print("点击「文字」Tab...")
    text_tab.first.click()
    _page.wait_for_timeout(1000)

# 标题
title_sel = 'input[placeholder*="标题"], textarea[placeholder*="标题"], [contenteditable][placeholder*="标题"]'
if _page.locator(title_sel).count() > 0:
    _page.locator(title_sel).first.click()
    _page.locator(title_sel).first.fill(NOTE_TITLE)
    print(f"✓ 标题: {NOTE_TITLE!r}")
else:
    print("⚠ 未找到标题输入框")

# 正文
body_sel = ('textarea[placeholder*="内容"], [contenteditable][placeholder*="内容"],'
            ' [contenteditable][placeholder*="正文"]')
if _page.locator(body_sel).count() > 0:
    _page.locator(body_sel).first.click()
    _page.locator(body_sel).first.fill(NOTE_BODY)
    print(f"✓ 正文: {NOTE_BODY!r}")
else:
    print("⚠ 未找到正文输入框")

print("\n内容已填写，请在浏览器中确认，之后运行 Cell 7 关闭页面。")

In [ ]:
# ── Cell 7 · 关闭页面 ────────────────────────────────────────────────
if _page and not _page.is_closed():
    _page.close()
    print("✓ 页面已关闭")
if _context:
    _context.close()
    print("✓ 上下文已关闭")

In [ ]:
# ── Cell 8 · 关闭服务器 ──────────────────────────────────────────────
if _browser:
    try:
        _browser.close()
    except Exception:
        pass
if _playwright:
    _playwright.stop()
if _server_proc and _server_proc.poll() is None:
    _server_proc.terminate()
    _server_proc.wait(timeout=5)
URL_FILE.unlink(missing_ok=True)
print("✓ 服务器已关闭，server_url.txt 已清理")